In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np

%load_ext autoreload
%autoreload 2

In [4]:
from src.data import load_raw, clean
from src.config import FEATURE_NAMES, CONSTANT_COLS, LABEL_COL, FAMILY_COL, PROCESSED_DIR

raw = load_raw()
cleaned, report = clean(raw)

for k, v in report.items():
    if k in ("family_counts_before", "family_counts_after"):
        print(f"{k}")
        for fam in ["Normal", "DoS", "Probe", "R2L", "U2R"]:
            print(f"   {fam:6s}: {v.get(fam, 0)}")
    
    elif not isinstance(v, dict):
        print(f"{k}: {v}")


raw_rows: 494021
exact_duplicates_removed: 348435
conflict_rows_removed: 2
same_family_diff_attack_removed: 1
final_rows: 145583
family_counts_before
   Normal: 97278
   DoS   : 391458
   Probe : 4107
   R2L   : 1126
   U2R   : 52
family_counts_after
   Normal: 87831
   DoS   : 54571
   Probe : 2130
   R2L   : 999
   U2R   : 52
dropped_constant_cols: ['num_outbound_cmds', 'is_host_login']


In [5]:
print(f"exact_duplicates_removed = {report['exact_duplicates_removed']:,}  (pilot ~348,435)")
print(f"final_rows               = {report['final_rows']:,}  (pilot ~145,586, within a few)")
print(f"U2R after cleaning       = {report['family_counts_after'].get('U2R', 0)}  (pilot 52)")

exact_duplicates_removed = 348,435  (pilot ~348,435)
final_rows               = 145,583  (pilot ~145,586, within a few)
U2R after cleaning       = 52  (pilot 52)


In [6]:
from src.config import FAMILIES

# Test 1: no duplicate feature rows remain
feature_cols_present = [c for c in FEATURE_NAMES if c in cleaned.columns]
assert cleaned.duplicated(subset=feature_cols_present).sum() == 0
print("no duplicate feature rows remain")

# Test 2: cleaning is idempotent
cleaned2, report2 = clean(cleaned)
assert len(cleaned2) == len(cleaned), f"{len(cleaned2)} != {len(cleaned)}"
print("cleaning is idempotent")

# Test 3: families are a subset of the 5 expected, with no NaN
assert set(cleaned[FAMILY_COL].unique()) <= set(FAMILIES)
assert not cleaned[FAMILY_COL].isna().any()
print(" families subset of 5, no NaN")

# Test 4: family counts sum to the row count
assert cleaned[FAMILY_COL].value_counts().sum() == len(cleaned)
print("family counts sum to row count")

no duplicate feature rows remain
cleaning is idempotent
 families subset of 5, no NaN
family counts sum to row count


In [7]:
def make_synthetic():
    base = {c: 0 for c in FEATURE_NAMES if c not in ("protocol_type", "service", "flag")}
    for c in ("protocol_type", "service", "flag"):
        base[c] = "a"
    base2 = {**base, "duration": 999}   # a different feature vector
    base3 = {**base, "duration": 555}   # another different feature vector
    rows = [
        {**base,  LABEL_COL: "neptune"},  # 0: exact dup of row 1        -> kept
        {**base,  LABEL_COL: "neptune"},  # 1: exact dup of row 0        -> removed
        {**base2, LABEL_COL: "normal"},   # 2: conflict with row 3       -> removed
        {**base2, LABEL_COL: "neptune"},  # 3: conflict with row 2       -> removed
        {**base3, LABEL_COL: "satan"},    # 4: same fam as row 5         -> kept
        {**base3, LABEL_COL: "nmap"},     # 5: same fam/features as row4 -> removed
    ]
    return pd.DataFrame(rows, columns=FEATURE_NAMES + [LABEL_COL])

syn_cleaned, syn_report = clean(make_synthetic())

assert syn_report["raw_rows"] == 6
assert syn_report["exact_duplicates_removed"] == 1        # row 1
assert syn_report["conflict_rows_removed"] == 2           # rows 2,3
assert syn_report["same_family_diff_attack_removed"] == 1 # row 5
assert syn_report["final_rows"] == 2                      # rows 0,4
assert syn_cleaned[LABEL_COL].tolist() == ["neptune", "satan"]
print("synthetic test passes")
print("remaining labels:", syn_cleaned[LABEL_COL].tolist())
print("remaining families:", syn_cleaned[FAMILY_COL].tolist())

synthetic test passes
remaining labels: ['neptune', 'satan']
remaining families: ['DoS', 'Probe']


In [8]:
summary = pd.DataFrame({
    "before": pd.Series(report["attack_counts_before"]),
    "after":  pd.Series(report["attack_counts_after"]),
}).fillna(0).astype(int)
summary["removed"] = summary["before"] - summary["after"]
summary = summary.sort_values("removed", ascending=False)

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(PROCESSED_DIR / "dedup_summary.csv")

print(summary)
print(f"\nSaved -> {PROCESSED_DIR / 'dedup_summary.csv'}")

                 before  after  removed
smurf            280790    641   280149
neptune          107201  51820    55381
normal            97278  87831     9447
back               2203    968     1235
satan              1589    906      683
portsweep          1040    415      625
ipsweep            1247    651      596
warezclient        1020    893      127
nmap                231    158       73
teardrop            979    918       61
pod                 264    205       59
land                 21     19        2
imap                 12     12        0
buffer_overflow      30     30        0
ftp_write             8      8        0
guess_passwd         53     53        0
perl                  3      3        0
multihop              7      7        0
loadmodule            9      9        0
phf                   4      4        0
rootkit              10     10        0
spy                   2      2        0
warezmaster          20     20        0

Saved -> /home/nightingale/My_Projects/

Experiment

In [9]:
from src.data import load_raw, clean, add_family
from src.validate import row_overlap
from src.config import FEATURE_NAMES, LABEL_COL, FAMILY_COL, SEED

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

# Load both versions for comparison
df_raw = load_raw()
df_raw_with_family = add_family(df_raw) # We need 'family' for stratification

df_clean, report = clean(df_raw)

In [11]:
def run_leakage_experiment(df, name):
    feature_cols = [c for c in FEATURE_NAMES if c in df.columns]
    X = df[feature_cols]
    y = df[FAMILY_COL]

    # 1. Stratified Split (80/20)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )
    # 2. Calculate Overlap
    overlap_count = row_overlap(X_train, X_test, feature_cols)
    overlap_pct = overlap_count / len(X_test)
    
    # 3. Encode (Demo Shortcut)
    # We concat to align columns, then split back
    X_all = pd.concat([X_train, X_test])
    X_all_enc = pd.get_dummies(X_all)
    X_train_enc = X_all_enc.iloc[:len(X_train)]
    X_test_enc = X_all_enc.iloc[len(X_train):]
    
    # 4. Train Decision Tree
    clf = DecisionTreeClassifier(random_state=SEED)
    clf.fit(X_train_enc, y_train)
    
    # 5. Score
    y_pred = clf.predict(X_test_enc)
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    
    # 6. Honest Score (Only for Leaked Data)
    # Filter out rows in test that are also in train
    honest_acc = np.nan
    honest_f1 = np.nan
    if overlap_count > 0:
        hash_train = pd.util.hash_pandas_object(X_train, index=False)
        hash_test = pd.util.hash_pandas_object(X_test, index=False)
        is_leaked = hash_test.isin(hash_train)
        honest_mask = ~is_leaked
        
        if honest_mask.sum() > 0:
            y_test_honest = y_test[honest_mask]
            y_pred_honest = y_pred[honest_mask]
            honest_acc = accuracy_score(y_test_honest, y_pred_honest)
            honest_f1 = f1_score(y_test_honest, y_pred_honest, average='macro')

    # Print Results
    print(f"\n=== {name} ===")
    print(f"Test Overlap:  {overlap_count:>6} rows ({overlap_pct:.2%})")
    print(f"Accuracy:      {acc:.4f}")
    print(f"Macro-F1:      {f1_macro:.4f}")
    if not np.isnan(honest_acc):
        print(f"Honest Acc:    {honest_acc:.4f} (on non-leaked rows only)")
        print(f"Honest F1:     {honest_f1:.4f}")
        
    return overlap_count, overlap_pct, acc, f1_macro, honest_acc, honest_f1

# Run on Raw (Leaked)
res_raw = run_leakage_experiment(df_raw_with_family, "RAW (Leaked)")

# Run on Clean (Safe)
res_clean = run_leakage_experiment(df_clean, "CLEAN (Safe)")

# Verification
assert res_clean[0] == 0, "Clean data must have 0 overlap!"



=== RAW (Leaked) ===
Test Overlap:   72640 rows (73.52%)
Accuracy:      0.9997
Macro-F1:      0.9083
Honest Acc:    0.9988 (on non-leaked rows only)
Honest F1:     0.9063

=== CLEAN (Safe) ===
Test Overlap:       0 rows (0.00%)
Accuracy:      0.9991
Macro-F1:      0.9085


NEWA CLESL

In [14]:
from src.data import load_raw, clean, split_xy, make_split, save_split, load_split
from src.validate import validate_split
from src.config import PROCESSED_DIR, SEED

# 1. Load clean data
df_clean, _ = clean(load_raw())

# 2. Extract X, y, meta
X, y, meta = split_xy(df_clean)
print(f"Clean data shape: {X.shape}")

# 3. Make split (stratified on family)
train_idx, test_idx = make_split(y, test_size=0.2, seed=SEED)
print(f"Train size: {len(train_idx)}  (pilot ~116,468)")
print(f"Test size:  {len(test_idx)}  (pilot ~29,117)")

# 4. Save to disk
save_split(train_idx, test_idx, n_rows=len(X), seed=SEED)


Clean data shape: (145583, 39)
Train size: 116466  (pilot ~116,468)
Test size:  29117  (pilot ~29,117)


In [15]:
# 1. Validate the split in memory
validate_split(X, y, train_idx, test_idx, meta=None)
print(" In-memory validation passed")

# 2. Reload from disk and verify
train_idx_loaded, test_idx_loaded, meta_loaded = load_split(out_dir=PROCESSED_DIR, n_rows=len(X))
assert np.array_equal(train_idx, train_idx_loaded)
assert np.array_equal(test_idx, test_idx_loaded)
print("Reloaded indices match saved indices exactly")

# 3. Validate the reloaded split
validate_split(X, y, train_idx_loaded, test_idx_loaded, meta_loaded)
print(" Reloaded validation passed")

 In-memory validation passed
Reloaded indices match saved indices exactly
 Reloaded validation passed


In [16]:
# Break 1: Move one test index into train (duplicate/overlap)
train_broken = np.append(train_idx, test_idx[0])
test_broken = test_idx
try:
    validate_split(X, y, train_broken, test_broken, meta=None)
    print("FAIL: Overlap not caught!")
except AssertionError as e:
    print(f"PASS: Overlap caught -> {e}")

# Break 2: Forget an index (union doesn't cover all rows)
train_short = train_idx[:-5]
try:
    validate_split(X, y, train_short, test_idx, meta=None)
    print("FAIL: Missing rows not caught!")
except AssertionError as e:
    print(f"PASS: Missing rows caught -> {e}")


PASS: Overlap caught -> Train and test overlap! 1 indices in common.
PASS: Missing rows caught -> Train+Test (145578) != total rows (145583)


In [17]:
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("=== TRAIN distribution ===")
print(y_train.value_counts())

print("\n=== TEST distribution (LOCKED) ===")
print(y_test.value_counts())
print(f"\nU2R in test set: {y_test.value_counts().get('U2R', 0)} (pilot ~10)")

=== TRAIN distribution ===
family
Normal    70264
DoS       43657
Probe      1704
R2L         799
U2R          42
Name: count, dtype: int64

=== TEST distribution (LOCKED) ===
family
Normal    17567
DoS       10914
Probe       426
R2L         200
U2R          10
Name: count, dtype: int64

U2R in test set: 10 (pilot ~10)


PART B PHASE  ## Part B: Train-only EDA,

In [ ]:
import sys 
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

%load_ext autoreload
%autoreload 2


from src.data import load_train
from src.config import (
        PROJECT_ROOT, FEATURE_NAMES, CATEGORICAL_COLS, CONSTANT_COLS,
        FAMILIES, FAMILY_COL, LABEL_COL,
)

FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)



X_train, y_train, meta_train = load_train()

train = X_train.copy()

train[FAMILY_COL] = y_train.values
train[LABEL_COL] = meta_train.values

num_cols = [c for c in X_train.columns if X_train[c].dtype.kind in  "iuf"]
print(f"X_train: {X_train.shape}  | numeric cols: {len(num_cols)}")


In [ ]:
#class distribution, log scale:

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

y_train.value_counts().reindex(FAMILIES).plot.bar(ax=axes[0], color="steelblue")
axes[0].set_yscale("log")
axes[0].set_title("Family Distribution (train, log scale)")
axes[0].set_ylabel("Count (log)")


meta_train.value_counts().plot.bar(ax=axes[1], color="coral")
axes[1].set_yscale("log")
axes[1].set_title("Attack-name distribution (train, log scale)")
axes[1].tick_params(axis="x", labelsize=6)


plt.tight_layout()
plt.savefig(FIG_DIR / "p6_01_class_distribution.png", dpi=120)
plt.show()


In [ ]:
ct = pd.crosstab(train["protocol_type"], train[FAMILY_COL], normalize="index")
ct = ct.reindex(columns=FAMILIES)

plt.figure(figsize=(8, 3))
sns.heatmap(ct, annot=True, fmt=".3f", cmap="viridis")
plt.title("protocol_type x family (row-normalized)")
plt.tight_layout()
plt.savefig(FIG_DIR / "p6_02_protocol_family.png", dpi=120)
plt.show()
print(ct)



In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=False)
for ax, fam in zip(axes, FAMILIES):
    sub = train.loc[train[FAMILY_COL] == fam, "service"]
    sub.value_counts().head(15).plot.bar(ax=ax, color="seagreen")
    ax.set_title(fam)
    ax.tick_params(axis="x", labelsize=6, rotation=60)
plt.suptitle("Top 15 services per family (train)")
plt.tight_layout()
plt.savefig(FIG_DIR / "p6_03_service_per_family.png", dpi=120)
plt.show()


In [ ]:
ct_flag = pd.crosstab(train["flag"], train[FAMILY_COL], normalize="index").reindex(columns=FAMILIES)
plt.figure(figsize=(9, 4))
sns.heatmap(ct_flag, annot=True, fmt=".3f", cmap="magma")
plt.title("flag × family (row-normalized)")
plt.tight_layout()
plt.savefig(FIG_DIR / "p6_04_flag_family.png", dpi=120)
plt.show()

In [ ]:
skew_targets = ["duration", "src_bytes", "dst_bytes", "count", "dst_host_count"]
stats = train[skew_targets].agg(["median", "max", lambda s: s.skew()]).T
stats.columns = ["median", "max", "skew"]
print(stats)

fig, axes = plt.subplots(1, len(skew_targets), figsize=(20, 4))
for ax, col in zip(axes, skew_targets):
    sns.boxplot(data=train, x=FAMILY_COL, y=np.log1p(train[col]), ax=ax,
                order=FAMILIES, palette="Set2")
    ax.set_title(f"log1p({col})")
    ax.tick_params(axis="x", labelsize=7)
plt.suptitle("log1p(features) by family")
plt.tight_layout()
plt.savefig(FIG_DIR / "p6_05_skewed_by_family.png", dpi=120)
plt.show()


In [ ]:
zero_share = (train[num_cols] == 0).mean().sort_values(ascending=False)
print("Columns that are >95% zero:")
print(zero_share[zero_share > 0.95])

# Which families hold the non-zeros for key content features?
content = ["hot", "num_failed_logins", "root_shell", "num_root",
           "num_file_creations", "num_shells", "num_access_files"]
present = [c for c in content if c in train.columns]
nonzero_by_fam = (train[present + [FAMILY_COL]].groupby(FAMILY_COL)[present]
                  .apply(lambda d: (d > 0).mean()))
print("\nShare of NON-zero content features by family:")
print(nonzero_by_fam.round(3))

In [ ]:
corr = train[num_cols].corr(method="pearson")

plt.figure(figsize=(14, 11))
sns.heatmap(corr, cmap="coolwarm", center=0, square=True,
            cbar_kws={"shrink": 0.6}, xticklabels=False, yticklabels=False)
plt.title("Pearson correlation, numeric train features")
plt.tight_layout()
plt.savefig(FIG_DIR / "p6_07_correlation.png", dpi=120)
plt.show()

# Programmatic list of |r| > 0.95 pairs
pairs = []
cols = corr.columns
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        r = corr.iloc[i, j]
        if abs(r) > 0.95:
            pairs.append((cols[i], cols[j], round(r, 4)))
pairs.sort(key=lambda t: -abs(t[2]))
print(f"\n{len(pairs)} pairs with |r| > 0.95:")
for a, b, r in pairs:
    print(f"  {r:+.4f}  {a}  <->  {b}")

In [ ]:
binary_cols = [c for c in X_train.columns if X_train[c].nunique() == 2]
print("Binary columns (nunique == 2):", binary_cols)

print("\nsu_attempted value counts in train:")
print(train["su_attempted"].value_counts().sort_index())

In [ ]:
svc_counts = train["service"].value_counts()
rare = svc_counts[svc_counts < 5]
print(f"service values occurring < 5 times in train: {len(rare)}")
print(rare.to_dict() if len(rare) <= 20 else rare.head(20))

In [ ]:
# (a) traffic-window features separate DoS/Probe
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["count", "srv_count", "dst_host_serror_rate"]):
    sns.boxplot(data=train, x=FAMILY_COL, y=np.log1p(train[col]), ax=ax,
                order=FAMILIES, palette="Set2")
    ax.set_title(f"log1p({col})")
plt.suptitle("Traffic-window features (DoS/Probe signal)")
plt.tight_layout(); plt.savefig(FIG_DIR / "p6_10a_traffic.png", dpi=120); plt.show()

# (b) content features separate R2L/U2R
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, ["hot", "num_root", "num_file_creations"]):
    sns.boxplot(data=train, x=FAMILY_COL, y=train[col], ax=ax,
                order=FAMILIES, palette="Set2")
    ax.set_title(col)
plt.suptitle("Content features (R2L/U2R signal)")
plt.tight_layout(); plt.savefig(FIG_DIR / "p6_10b_content.png", dpi=120); plt.show()

In [ ]:
# BINARY: exactly 2 unique values
BINARY_COLS = sorted([c for c in X_train.columns if X_train[c].nunique() == 2])

# NUMERIC: every numeric feature that isn't binary (derived, not hand-typed)
NUMERIC_COLS = sorted([c for c in num_cols if c not in BINARY_COLS])

# HEAVY_TAILED: rule = numeric feature with skewness > 5.0
skew = train[NUMERIC_COLS].skew().sort_values(ascending=False)
HEAVY_TAIL_THRESHOLD = 5.0
HEAVY_TAILED_COLS = sorted(skew[skew > HEAVY_TAIL_THRESHOLD].index.tolist())

# Sanity: HEAVY_TAILED must be a subset of NUMERIC
assert set(HEAVY_TAILED_COLS) <= set(NUMERIC_COLS), "heavy-tailed must be numeric"

print("BINARY_COLS =", BINARY_COLS)
print("\nHEAVY_TAILED_COLS (skew > 5) =", HEAVY_TAILED_COLS)
print("\nNUMERIC_COLS count =", len(NUMERIC_COLS))
print("\nTop skewness values:")
print(skew.head(15))

# Correlated pairs to log (from Cell B7)
CORRELATED_PAIRS = [(a, b, r) for a, b, r in pairs]


In [ ]:
# These come from load_train, so they reflect the saved split
assert len(X_train) == 116468 or len(X_train) > 0  # shape sanity; compare to your split size
# The real guard: no train row is a test row
from src.data import load_split
train_idx, test_idx, _ = load_split()
assert len(set(train_idx).intersection(set(test_idx))) == 0, "train index leaks into test!"

# Coverage: CONSTANT + CATEGORICAL + BINARY + NUMERIC == all features
covered = set(CONSTANT_COLS) | set(CATEGORICAL_COLS) | set(BINARY_COLS) | set(NUMERIC_COLS)
missing = set(FEATURE_NAMES) - covered
assert not missing, f"Uncovered feature columns: {missing}"
print(" coverage: every feature column lands in exactly one group")
print(f"  CONSTANT={len(CONSTANT_COLS)}, CATEGORICAL={len(CATEGORICAL_COLS)}, "
      f"BINARY={len(BINARY_COLS)}, NUMERIC={len(NUMERIC_COLS)}")
